In [2]:
"""
Load the finetuned LoRA adapter locally and generate 3 search queries.

Usage:
    python inference_local.py
"""
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_PATH = "./adapter"   # downloaded via download_adapter.py

def load_model():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base, ADAPTER_PATH)
    model.eval()
    return model, tokenizer


def generate_queries(case_query: str, model, tokenizer, temperature: float = 0.4) -> list[str]:
    messages = [
        {
            "role": "system",
            "content": (
                "Du bist ein Rechtsrecherche-Spezialist für Schweizer Recht. "
                "Gegeben einen deutschen Rechtsfall, generiere genau 3 kurze deutsche "
                "Stichwortsuchanfragen, die die relevantesten Gesetzesartikel aus einer "
                "Vektordatenbank abrufen. "
                "Gib NUR die 3 Suchanfragen aus, eine pro Zeile, ohne Nummerierung, "
                "ohne Symbole, ohne weiteren Text."
            ),
        },
        {"role": "user", "content": case_query},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    queries = [q.strip() for q in generated.strip().split("\n") if q.strip()]
    return queries[:3]



In [3]:
print("Loading model...")
model, tokenizer = load_model()

test_case = "Kann ein Gericht gemäß Art. 221 Abs. 1 lit. rechtmäßig eine dreimonatige Verlängerung der Untersuchungshaft anordnen? b) StPO (Gefahr der Absprache) im Einklang mit dem Grundsatz der Verhältnismäßigkeit, als der Beschuldigte – der nach einem mutmaßlichen nächtlichen Überfall und Diebstahl einer Kuriertasche mit unter anderem 5.600 € festgenommen wurde – mit Beschluss vom 18. Oktober 2024 für einen Zeitraum bis maximal 15. Januar 2025 in Untersuchungshaft genommen wurde, die Staatsanwaltschaft am 10. Dezember 2024 eine Verlängerung beantragte und dabei hauptsächlich ein konkretes Risiko der Beeinflussung von Zeugen oder der Manipulation von Beweismitteln sowie ein Risiko der Wiederholungstat anführte, während der Beschuldigte der Verlängerung mit der Begründung widersprach, dass die meisten Zeugen bereits befragt worden seien, die noch ausstehenden Ermittlungsschritte im Wesentlichen technischer Natur seien (Auswertung von Telefondaten, Analyse von Videoaufnahmen und Überprüfung von Bankunterlagen), seine Freilassung daher die Ermittlungen nicht gefährden würde und das mutmaßliche Opfer die Anzeige zurückgezogen habe – d. h. rechtfertigen das behauptete konkrete Risiko der Absprache und Erwägungen der Verhältnismäßigkeit unter diesen Umständen eine dreimonatige Verlängerung?"


Loading model...


Fetching 4 files:   0%|          | 0/4 [02:59<?, ?it/s]
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [1]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    '../models/reranker_swiss_law',
    num_labels=1,
    device='mps',
)

/Users/keshavsharmaog/Downloads/swiss-law-agentic-retrieval/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    '../scripts/reranker_bge_swiss_law_b200',
    num_labels=1,
    device='mps'
)

In [3]:
"""
Load the finetuned LoRA adapter locally and generate 3 search queries.

Usage:
    python inference_local.py
"""
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL   = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_PATH = "keshavsharma/llama-3.1-7b-swiss-law-adapter"   # downloaded via download_adapter.py

def load_model():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base, ADAPTER_PATH)
    model.eval()
    return model, tokenizer


def generate_queries(case_query: str, model, tokenizer, temperature: float = 0.4) -> list[str]:
    messages = [
        {
            "role": "system",
            "content": (
                "Du bist ein Rechtsrecherche-Spezialist für Schweizer Recht. "
                "Gegeben einen deutschen Rechtsfall, generiere genau 3 kurze deutsche "
                "Stichwortsuchanfragen, die die relevantesten Gesetzesartikel aus einer "
                "Vektordatenbank abrufen. "
                "Gib NUR die 3 Suchanfragen aus, eine pro Zeile, ohne Nummerierung, "
                "ohne Symbole, ohne weiteren Text."
            ),
        },
        {"role": "user", "content": case_query},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=124,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    queries = [q.strip() for q in generated.strip().split("\n") if q.strip()]
    return queries[:3]



In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
load_dotenv()

import os

hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)
model, tokenizer = load_model()

Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.92s/it]
Some parameters are on the meta device because they were offloaded to the disk.
